# 🛡️ Sensitive Data Leak Detector for APIs (T1-PS7)
### Interactive Evaluation & Demonstration Notebook

This notebook demonstrates the end-to-end workflow of the **Sensitive Data Leak Detector for APIs** according to the Software Requirements Specification (SRS):
1. **Load sample API responses** (JSON, JSON Lines, plain text logs)
2. **Run detection engine** with built-in regex rules, entropy checks, and Luhn validation
3. **Inspect classified findings** with severity rankings, locations, and compliance mappings
4. **View sanitized / masked output** where raw secrets are completely redacted
5. **Export reports** into Markdown, HTML, JSON, and CSV

## Step 1: Import Modules & Initialize Engine

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from leakdetect.loader import load_file, load_string, load_folder
from leakdetect.detector import LeakDetector
from leakdetect.reporter import LeakReporter
from leakdetect.classifier import calculate_overall_risk
import pandas as pd

rules_path = root_dir / "rules" / "default_rules.yaml"
detector = LeakDetector(rules_path=rules_path)
print(f"[*] Loaded {len(detector.rules)} detection rules from {rules_path.name}")
print(f"[*] Configured Allow-List: {sorted(list(detector.allow_list))}")

## Step 2: Load and Scan Leaky User PII Response
Testing contact PII, Indian Aadhaar numbers, and PAN card numbers.

In [ ]:
pii_file = root_dir / "samples" / "leaky_user_pii.json"
item = load_file(pii_file)
findings, masked_payload = detector.scan_item(item)

print(f"[*] Scanned: {item.source_name}")
print(f"[*] Detected {len(findings)} sensitive leak finding(s).")

# Display findings as formatted Pandas table
df_findings = pd.DataFrame([f.to_dict() for f in findings])
display_cols = ["finding_id", "location", "type", "severity", "masked_value", "rule_name", "recommendation"]
df_findings[display_cols]

## Step 3: View Masked / Sanitized Payload
Observe how raw emails, phone numbers, Aadhaar, and PAN have been replaced safely while keeping valid JSON structure.

In [ ]:
print("=== SANITIZED API PAYLOAD (RAW SECRETS NEVER LEAKED) ===")
print(masked_payload)

## Step 4: Scan Credentials & Cryptographic Keys
Testing detection of passwords, bcrypt hashes, JWTs, Bearer headers, AWS keys, and RSA private keys.

In [ ]:
auth_file = root_dir / "samples" / "leaky_auth_tokens.json"
auth_item = load_file(auth_file)
auth_findings, _ = detector.scan_item(auth_item)

df_auth = pd.DataFrame([f.to_dict() for f in auth_findings])
df_auth[["finding_id", "location", "severity", "masked_value", "rule_name", "risk"]]

## Step 5: Test Financial Data & Luhn Validation (TC-08)
Validating that valid credit card numbers are flagged while numbers failing the Luhn checksum are rejected.

In [ ]:
card_file = root_dir / "samples" / "leaky_payment_cards.json"
card_item = load_file(card_file)
card_findings, _ = detector.scan_item(card_item)

pd.DataFrame([f.to_dict() for f in card_findings])[["finding_id", "location", "severity", "masked_value", "recommendation"]]

## Step 6: Test Clean API Response (TC-10)
Ensuring zero false positives on compliant payloads.

In [ ]:
clean_file = root_dir / "samples" / "clean_user_response.json"
clean_item = load_file(clean_file)
clean_findings, _ = detector.scan_item(clean_item)

risk = calculate_overall_risk([f.to_dict() for f in clean_findings])
print(f"[*] Clean Payload Findings: {len(clean_findings)}")
print(f"[*] Overall Risk Status   : {risk['level']} (Score: {risk['score']}/100)")

## Step 7: Export Comprehensive Scan Report
Generating report exports across HTML, Markdown, and CSV.

In [ ]:
reporter = LeakReporter(
    findings=findings + auth_findings + card_findings,
    files_scanned=["leaky_user_pii.json", "leaky_auth_tokens.json", "leaky_payment_cards.json"],
)

out_dir = root_dir / "reports"
out_dir.mkdir(exist_ok=True)

reporter.save_report(out_dir / "demo_report.md", format_type="md")
reporter.save_report(out_dir / "demo_report.html", format_type="html")
reporter.save_report(out_dir / "demo_report.csv", format_type="csv")
reporter.save_report(out_dir / "demo_report.json", format_type="json")

print(f"[+] Reports saved successfully to {out_dir.resolve()}")